In [ ]:
import pandas as pd
import os
from pathlib import Path

BASE = Path().resolve()
if BASE.name == 'notebooks':
    BASE = BASE.parent

df = pd.read_csv(BASE / 'data/processed/raw_merged.csv')


In [ ]:
df = df.dropna(subset=['Q1_s', 'grid_position', 'finish_position'])

# q_session_reached must be before filling zeros
df['q_session_reached'] = 1
df.loc[df['Q2_s'].notna(), 'q_session_reached'] = 2
df.loc[df['Q3_s'].notna(), 'q_session_reached'] = 3

df['Q2_s'] = df['Q2_s'].fillna(0.0)
df['Q3_s'] = df['Q3_s'].fillna(0.0)


In [ ]:
# delta to pole, drivers best time minus the pole time in that race
df['best_time'] = df['Q3_s'].replace(0, float('nan'))
df['best_time'] = df['best_time'].fillna(df['Q2_s'].replace(0, float('nan')))
df['best_time'] = df['best_time'].fillna(df['Q1_s'])

q3_clean = df['Q3_s'].replace(0, float('nan'))
pole_times = q3_clean.groupby([df['season'], df['round']]).min()
df['pole_time'] = df.set_index(['season', 'round']).index.map(pole_times)
df['delta_to_pole_s'] = df['best_time'] - df['pole_time']
df = df.drop(columns=['best_time', 'pole_time'])


In [ ]:
cols = ['season', 'grid_position', 'Q1_s', 'Q2_s', 'Q3_s',
        'delta_to_pole_s', 'q_session_reached', 'finish_position']

os.makedirs(BASE / 'data/processed', exist_ok=True)
df[cols].to_csv(BASE / 'data/processed/clean_dataset.csv', index=False)

print(f'Saved: {df[cols].shape}')
print(df[cols].head())
